# 1. Configuração (Credenciais do Supabase)

In [0]:
import pandas as pd
import psycopg2 

# --------------------------------------------------------------
# 1. Configuração (Credenciais do Supabase)
# --------------------------------------------------------------
DB_SERVER   = "aws-1-us-east-1.pooler.supabase.com"
DB_PORT     = "6543"
DB_USER     = "postgres.znhpaegcmcbrgbncxapt"
DB_PASSWORD = "abacaxiRabanete" 
DB_DATABASE = "postgres"
# AQUI ESTAVA O SEGREDO: Mudamos para o schema correto!
DB_SCHEMA   = "public"

print(f'Conectando ao Supabase ({DB_SERVER})...')

# 2. Conexão com Supabase (PostgreSQL)

In [0]:
try:
    conn = psycopg2.connect(
        host=DB_SERVER, 
        port=DB_PORT,
        database=DB_DATABASE, 
        user=DB_USER,
        password=DB_PASSWORD
    )
    cursor = conn.cursor()
    print('Conectado com sucesso ao banco de dados!\n')
except Exception as e:
    print(f'Erro na conexão: {e}')

# 3. Listar Tabelas do Schema 'sistema_ouvidoria'

In [0]:
cursor.execute(f"""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = '{DB_SCHEMA}' 
    AND table_type = 'BASE TABLE' 
""")
tabelas = [row[0] for row in cursor.fetchall()]
print(f'Iniciando extração de {len(tabelas)} tabelas...\n')

# 4. Extrair Tabelas -> CSV no Databricks (DBFS)

In [0]:
for tabela in tabelas:
    query = f'SELECT * FROM "{DB_SCHEMA}"."{tabela}"'
    df_pandas = pd.read_sql(query, conn)

    if df_pandas.empty:
        print(f'  Ignorando {tabela} (0 registros - Tabela Vazia)')
        continue

    df_spark = spark.createDataFrame(df_pandas)

    nome_tabela_destino = f"workspace.landing.{tabela}"
    df_spark.write.mode("overwrite").saveAsTable(nome_tabela_destino)
    
    print(f'  ✓ {tabela} salva em {nome_tabela_destino} ({len(df_pandas)} linhas)')

# 5. Validação e Resumo

In [0]:
# Fechar conexões
cursor.close()
conn.close()
print('\nExtração 100% concluída! Suas tabelas já estão no Databricks prontas para a próxima fase.')